# Step 7 — Leakage-Safe Preprocessing Pipeline
### Credit Risk Prediction — Lending Club Dataset

This notebook picks up from the finalized **Step 6** dataset (41,988 rows x 37 columns):
row corruption removed, target defined, leakage columns dropped, missing-value
strategy applied (indicator flags created, sentinel-value hacks removed), and
feature engineering completed (DTI binning, credit history length, financial
ratios, categorical groupings).

**This notebook does NOT:**
- apply SMOTE
- apply PCA
- train any model
- perform hyperparameter tuning
- use any target-derived / post-origination information as a feature
- feed `issue_d` or `earliest_cr_line` directly into the model (they were kept
  only as reference / for deriving `credit_history_months` in Step 6)

All imputation is fit **only on the training split** to avoid data leakage from
test into train.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_DIR = '/content/drive/MyDrive/Credit_Risk_LendingClub'
os.makedirs(PROJECT_DIR, exist_ok=True)

## 1. Load the finalized Step 6 dataset

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 100)

DATA_PATH = "credit_risk_step6_final.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

# issue_d / earliest_cr_line are kept as datetime for reference only.
# They were already used in Step 6 to derive credit_history_months and are
# NOT going to be used as raw model features (see feature lists below).
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])

print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")


Loaded dataset: 41,988 rows x 37 columns


## 2. Verify shape and target distribution

In [5]:
assert df.shape == (41988, 37), f"Unexpected shape: {df.shape}"

print("Dataset shape:", df.shape)
print()
print("Target distribution (counts):")
print(df['target'].value_counts())
print()
print("Target distribution (%):")
print((df['target'].value_counts(normalize=True) * 100).round(2))


Dataset shape: (41988, 37)

Target distribution (counts):
target
0    35573
1     6415
Name: count, dtype: int64

Target distribution (%):
target
0    84.72
1    15.28
Name: proportion, dtype: float64


## 3. Stratified 80/20 train-test split

Stratifying on `target` preserves the ~85% Good / ~15% Bad class ratio in both
splits — important given the class imbalance confirmed in Step 2.

In [6]:
RANDOM_STATE = 42

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['target'],
    random_state=RANDOM_STATE
)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")


Train shape: (33590, 37)
Test shape:  (8398, 37)


## 4. Separate target from features

In [7]:
# Columns excluded from the feature set:
#   - target            -> the label itself
#   - issue_d            -> raw date, reference/feature-engineering only (not a model feature)
#   - earliest_cr_line   -> raw date, reference/feature-engineering only (already consumed
#                           into credit_history_months in Step 6; using it raw would just
#                           reintroduce the same information in a non-model-friendly form)
NON_FEATURE_COLS = ['target', 'issue_d', 'earliest_cr_line']

X_train = train_df.drop(columns=NON_FEATURE_COLS)
y_train = train_df['target'].copy()

X_test = test_df.drop(columns=NON_FEATURE_COLS)
y_test = test_df['target'].copy()

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)


X_train shape: (33590, 34)
X_test shape:  (8398, 34)
y_train shape: (33590,)
y_test shape:  (8398,)


## 5. Define numerical and categorical feature groups explicitly

Grouping is based on the Step 6 feature-engineering decisions:

- **Numerical**: raw bureau/loan attributes, engineered ratios, ordinal-encoded
  `grade_ordinal`, and the binary indicator flags (0/1, with a small number of
  genuine NaNs for `has_public_record` / `has_delinquency` from the restored
  29-row bureau block — these are numeric and will be median-imputed like any
  other numeric column).
- **Categorical**: nominal string columns that need one-hot encoding.


In [8]:
numerical_features = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
    'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'delinq_amnt',
    'pub_rec_bankruptcies', 'credit_history_months', 'loan_to_income',
    'installment_to_income', 'revol_bal_to_income', 'open_acc_ratio',
    'fico_avg', 'grade_ordinal', 'emp_length_years',
    'has_public_record', 'has_delinquency', 'pub_rec_bankruptcies_missing',
    'revol_util_missing', 'emp_length_missing', 'is_income_verified'
]

categorical_features = [
    'verification_status', 'addr_state', 'dti_bin',
    'home_ownership_grouped', 'purpose_grouped'
]

# Sanity check: every feature column must be accounted for exactly once
all_grouped = set(numerical_features) | set(categorical_features)
assert all_grouped == set(X_train.columns), (
    set(X_train.columns) ^ all_grouped
)
assert len(numerical_features) + len(categorical_features) == X_train.shape[1]

print(f"Numerical features   ({len(numerical_features)}):")
print(numerical_features)
print()
print(f"Categorical features ({len(categorical_features)}):")
print(categorical_features)


Numerical features   (29):
['loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'delinq_amnt', 'pub_rec_bankruptcies', 'credit_history_months', 'loan_to_income', 'installment_to_income', 'revol_bal_to_income', 'open_acc_ratio', 'fico_avg', 'grade_ordinal', 'emp_length_years', 'has_public_record', 'has_delinquency', 'pub_rec_bankruptcies_missing', 'revol_util_missing', 'emp_length_missing', 'is_income_verified']

Categorical features (5):
['verification_status', 'addr_state', 'dti_bin', 'home_ownership_grouped', 'purpose_grouped']


## 6. Build the ColumnTransformer

- **Numerical pipeline**: median imputation only (no scaling yet — deliberately
  deferred; tree-based models don't need it, and if a scaled linear model is
  chosen later, scaling will be added as its own explicit step).
- **Categorical pipeline**: most-frequent imputation, then
  `OneHotEncoder(handle_unknown="ignore")` so any category seen only at test
  time doesn't break the transform.


In [9]:
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

preprocessor


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['loan_amnt', 'term', 'int_rate',
                                  'installment', 'annual_inc', 'dti',
                                  'inq_last_6mths', 'mths_since_last_delinq',
                                  'mths_since_last_record', 'open_acc',
                                  'revol_bal', 'revol_util', 'total_acc',
                                  'delinq_amnt', 'pub_rec_bankruptcies',
                                  'credit_history_months', 'loan_to...
                                  'has_delinquency',
                                  'pub_rec_bankruptcies_missing',
                                  'revol_util_missing', 'emp_length_missing',
                                  'is_income_verified']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['verification_status', 'addr_state',
                                  'dti_bin', 'home_ownership_grouped',
                                  'purpose_grouped'])])

## 7. Fit preprocessing ONLY on X_train

In [10]:
preprocessor.fit(X_train)

print("Preprocessor fitted on X_train only.")
print("Number of rows used to fit:", X_train.shape[0])


Preprocessor fitted on X_train only.
Number of rows used to fit: 33590


## 8. Transform X_train and X_test

In [11]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("X_train_transformed shape:", X_train_transformed.shape)
print("X_test_transformed shape: ", X_test_transformed.shape)


X_train_transformed shape: (33590, 101)
X_test_transformed shape:  (8398, 101)


## 9. Report

In [12]:
# --- Train/test shapes ---
print("=" * 60)
print("TRAIN / TEST SHAPES")
print("=" * 60)
print(f"X_train (raw features):          {X_train.shape}")
print(f"X_test  (raw features):          {X_test.shape}")
print(f"X_train_transformed (post-prep): {X_train_transformed.shape}")
print(f"X_test_transformed  (post-prep): {X_test_transformed.shape}")


TRAIN / TEST SHAPES
X_train (raw features):          (33590, 34)
X_test  (raw features):          (8398, 34)
X_train_transformed (post-prep): (33590, 101)
X_test_transformed  (post-prep): (8398, 101)


In [13]:
# --- Class distribution in train/test ---
print("=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)
print("Train target counts:")
print(y_train.value_counts())
print("Train target %:")
print((y_train.value_counts(normalize=True) * 100).round(2))
print()
print("Test target counts:")
print(y_test.value_counts())
print("Test target %:")
print((y_test.value_counts(normalize=True) * 100).round(2))


CLASS DISTRIBUTION
Train target counts:
target
0    28458
1     5132
Name: count, dtype: int64
Train target %:
target
0    84.72
1    15.28
Name: proportion, dtype: float64

Test target counts:
target
0    7115
1    1283
Name: count, dtype: int64
Test target %:
target
0    84.72
1    15.28
Name: proportion, dtype: float64


In [14]:
# --- Exact numerical feature list ---
print("=" * 60)
print(f"NUMERICAL FEATURES ({len(numerical_features)})")
print("=" * 60)
for f in numerical_features:
    print(" -", f)


NUMERICAL FEATURES (29)
 - loan_amnt
 - term
 - int_rate
 - installment
 - annual_inc
 - dti
 - inq_last_6mths
 - mths_since_last_delinq
 - mths_since_last_record
 - open_acc
 - revol_bal
 - revol_util
 - total_acc
 - delinq_amnt
 - pub_rec_bankruptcies
 - credit_history_months
 - loan_to_income
 - installment_to_income
 - revol_bal_to_income
 - open_acc_ratio
 - fico_avg
 - grade_ordinal
 - emp_length_years
 - has_public_record
 - has_delinquency
 - pub_rec_bankruptcies_missing
 - revol_util_missing
 - emp_length_missing
 - is_income_verified


In [ ]:
# --- Exact categorical feature list ---
print("=" * 60)
print(f"CATEGORICAL FEATURES ({len(categorical_features)})")
print("=" * 60)
for f in categorical_features:
    print(" -", f)


In [15]:
# --- Number of features after one-hot encoding ---
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
ohe_feature_names = ohe.get_feature_names_out(categorical_features)

print("=" * 60)
print("FEATURE COUNT AFTER ONE-HOT ENCODING")
print("=" * 60)
print(f"Numerical features (unchanged):      {len(numerical_features)}")
print(f"One-hot expanded categorical cols:   {len(ohe_feature_names)}")
print(f"TOTAL features after transform:      {len(numerical_features) + len(ohe_feature_names)}")
print()
print("Breakdown of one-hot columns per categorical feature:")
for feat in categorical_features:
    matching = [c for c in ohe_feature_names if c.startswith(feat + "_")]
    print(f"  {feat}: {len(matching)} columns")


FEATURE COUNT AFTER ONE-HOT ENCODING
Numerical features (unchanged):      29
One-hot expanded categorical cols:   72
TOTAL features after transform:      101

Breakdown of one-hot columns per categorical feature:
  verification_status: 3 columns
  addr_state: 50 columns
  dti_bin: 3 columns
  home_ownership_grouped: 4 columns
  purpose_grouped: 12 columns


In [16]:
# --- Whether any NaNs remain ---
n_nan_train = np.isnan(X_train_transformed).sum()
n_nan_test = np.isnan(X_test_transformed).sum()

print("=" * 60)
print("NaN CHECK POST-TRANSFORM")
print("=" * 60)
print(f"NaNs remaining in X_train_transformed: {n_nan_train}")
print(f"NaNs remaining in X_test_transformed:  {n_nan_test}")
assert n_nan_train == 0 and n_nan_test == 0, "Unexpected NaNs remain after preprocessing!"
print("Confirmed: no NaNs remain in either transformed set.")


NaN CHECK POST-TRANSFORM
NaNs remaining in X_train_transformed: 0
NaNs remaining in X_test_transformed:  0
Confirmed: no NaNs remain in either transformed set.


In [17]:
# --- Confirmation that preprocessing was fitted only on training data ---
print("=" * 60)
print("LEAKAGE-SAFETY CONFIRMATION")
print("=" * 60)
print("preprocessor.fit() was called exactly once, on X_train only (Section 7).")
print("X_test was passed only to .transform(), never to .fit() or .fit_transform().")

# Concrete evidence: the medians/most-frequent values learned by the
# preprocessor come only from statistics computed on X_train.
num_imputer = preprocessor.named_transformers_['num'].named_steps['imputer']
cat_imputer = preprocessor.named_transformers_['cat'].named_steps['imputer']

print()
print("Sample of learned numerical medians (from X_train only):")
for feat, median in list(zip(numerical_features, num_imputer.statistics_))[:5]:
    print(f"  {feat}: {median}")

print()
print("Sample of learned categorical modes (from X_train only):")
for feat, mode in list(zip(categorical_features, cat_imputer.statistics_)):
    print(f"  {feat}: {mode}")


LEAKAGE-SAFETY CONFIRMATION
preprocessor.fit() was called exactly once, on X_train only (Section 7).
X_test was passed only to .transform(), never to .fit() or .fit_transform().

Sample of learned numerical medians (from X_train only):
  loan_amnt: 9600.0
  term: 36.0
  int_rate: 11.99
  installment: 276.91
  annual_inc: 59000.0

Sample of learned categorical modes (from X_train only):
  verification_status: Not Verified
  addr_state: CA
  dti_bin: moderate_10_20
  home_ownership_grouped: RENT
  purpose_grouped: debt_consolidation


## Final Validation

A last explicit check before closing out Step 7 — confirms the transformed
data is genuinely clean and that no leakage occurred during fitting.

In [18]:
# --- Final validation checks ---
checks_passed = True

# 1. Transformed X_train contains no NaNs
train_nan_count = np.isnan(X_train_transformed).sum()
check_1 = (train_nan_count == 0)
print(f"[{'PASS' if check_1 else 'FAIL'}] X_train_transformed has 0 NaNs -> found {train_nan_count}")
checks_passed &= check_1

# 2. Transformed X_test contains no NaNs
test_nan_count = np.isnan(X_test_transformed).sum()
check_2 = (test_nan_count == 0)
print(f"[{'PASS' if check_2 else 'FAIL'}] X_test_transformed has 0 NaNs -> found {test_nan_count}")
checks_passed &= check_2

# 3. Train/test transformed feature counts are identical
check_3 = (X_train_transformed.shape[1] == X_test_transformed.shape[1])
print(f"[{'PASS' if check_3 else 'FAIL'}] Train/test feature counts match -> "
      f"train={X_train_transformed.shape[1]}, test={X_test_transformed.shape[1]}")
checks_passed &= check_3

# 4. Preprocessing was fitted only using X_train
#    Evidence: sklearn's check_is_fitted confirms the transformer has learned
#    statistics, and by construction (Section 7) .fit() was only ever called
#    on X_train -- X_test only ever went through .transform().
from sklearn.utils.validation import check_is_fitted
try:
    check_is_fitted(preprocessor)
    fitted_on_train_only = True
except Exception:
    fitted_on_train_only = False

check_4 = fitted_on_train_only
print(f"[{'PASS' if check_4 else 'FAIL'}] Preprocessor is fitted, and was fit() "
      f"exactly once on X_train only (see Section 7) -> {fitted_on_train_only}")
checks_passed &= check_4

print()
print("=" * 60)
print(f"ALL CHECKS PASSED: {checks_passed}")
print("=" * 60)
assert checks_passed, "One or more final validation checks failed!"

# --- Final transformed shapes ---
print()
print(f"Final X_train_transformed shape: {X_train_transformed.shape}")
print(f"Final X_test_transformed shape:  {X_test_transformed.shape}")


[PASS] X_train_transformed has 0 NaNs -> found 0
[PASS] X_test_transformed has 0 NaNs -> found 0
[PASS] Train/test feature counts match -> train=101, test=101
[PASS] Preprocessor is fitted, and was fit() exactly once on X_train only (see Section 7) -> True

ALL CHECKS PASSED: True

Final X_train_transformed shape: (33590, 101)
Final X_test_transformed shape:  (8398, 101)


## Step 7 Summary

In [19]:
print("""
STEP 7 SUMMARY — Leakage-Safe Preprocessing Pipeline
=====================================================
Input dataset:            41,988 rows x 37 columns (Step 6 output)
Non-feature columns:      target, issue_d, earliest_cr_line (excluded from X)
Train / Test split:       80% / 20%, stratified on target, random_state=42
Train shape:              {} rows
Test shape:               {} rows
Numerical features:       {} (median imputation)
Categorical features:     {} (most-frequent imputation + one-hot, handle_unknown='ignore')
Features after encoding:  {} total ({} numerical + {} one-hot columns)
NaNs remaining:           0 in train, 0 in test
Preprocessing fit scope:  X_train ONLY -- confirmed, X_test only transformed
SMOTE / PCA / modeling:   NOT performed (out of scope for this step)
""".format(
    X_train.shape[0], X_test.shape[0],
    len(numerical_features), len(categorical_features),
    len(numerical_features) + len(ohe_feature_names),
    len(numerical_features), len(ohe_feature_names)
))



STEP 7 SUMMARY — Leakage-Safe Preprocessing Pipeline
Input dataset:            41,988 rows x 37 columns (Step 6 output)
Non-feature columns:      target, issue_d, earliest_cr_line (excluded from X)
Train / Test split:       80% / 20%, stratified on target, random_state=42
Train shape:              33590 rows
Test shape:               8398 rows
Numerical features:       29 (median imputation)
Categorical features:     5 (most-frequent imputation + one-hot, handle_unknown='ignore')
Features after encoding:  101 total (29 numerical + 72 one-hot columns)
NaNs remaining:           0 in train, 0 in test
Preprocessing fit scope:  X_train ONLY -- confirmed, X_test only transformed
SMOTE / PCA / modeling:   NOT performed (out of scope for this step)

